In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"

In [2]:
from datasets import load_dataset
from nnsight import LanguageModel
from typing import List
from sacrebleu.metrics import BLEU, CHRF
from tqdm import tqdm
import torch
from vllm import LLM, SamplingParams

In [ ]:
seperator = "\n"
transition = "->"
transisition_token = "->"
preprompt = "Examples:"
nshot = 10
model = "meta-llama/Llama-2-7b-hf"

# Load Dataset

In [4]:
ds_flores = load_dataset("facebook/flores", "all")
lang1, lang2 = "eng_Latn", "fra_Latn"

pairs = ds_flores["dev"].map(
    lambda x: {"text": x["sentence_" + lang1] + transition + x["sentence_" + lang2]}
)["text"]

In [5]:
ds_anto = load_dataset("kh4dien/synonym-antonym", split="train").filter(
    lambda x: x["type"] == "antonym"
)

pairs = ds_anto.shuffle(seed=42).map(
    lambda x: {"text": x["input"] + transition + x["output"]}
)["text"]

# Datasets functions

In [6]:
def get_n_shot(
    samples: List[str], n: int, preprompt: str, sep: str, transition: str
) -> List[dict]:
    """
    Generate n-shot examples for the given language pair.
    Args:
        lang1 (str): The source language code.
        lang2 (str): The target language code.
        sep (str): The separator between the two languages in the text.
        n (int): The number of examples to include in the prompt before the query.
    Returns:
        str: A string containing n-shot examples followed by a new input prompt.
    """

    result = []

    for i in range(len(samples) // n):
        sample = {
            "context": preprompt + sep,
        }

        for j in range(n - 1):
            sample["context"] += samples[i * n + j] + sep

        sample["query"] = samples[i * n + n - 1].split(transition)[0]
        sample["reference"] = samples[i * n + n - 1].split(transition)[1]

        sample["nshot_input"] = (
            sample["context"] + samples[i * n + n - 1].split(transition)[0] + transition
        )

        sample["0shot_input"] = (
            preprompt + sep + samples[i * n + n - 1].split(transition)[0] + transition
        ).strip()

        result.append(sample)

    return result


samples = get_n_shot(pairs, nshot, preprompt, seperator, transition)

samples[0], len(samples)

({'context': 'Examples:\nelevation->depression\nlimitless->limited\nluxurious->basic\nimmature->mature\nidiot->genius\nunavailable->available\ninner->outer\nadolescent->elderly\nnominal->real\nblend->stand out\nrelevant->irrelevant\nsober->drunk\nbrethren->sistren\noutfield->infield\nentire->partial\nwise->foolish\nrainy->sunny\npermit->prohibit\npopular->unpopular\nviable->unviable\nsupporter->opponent\nescalate->deescalate\nmobile->stationary\nclosing->opening\nskew->straight\npowerless->powerful\naccompany->leave alone\nsevere->mild\ngraduating->enrolling\n',
  'query': 'commence',
  'reference': 'conclude',
  'nshot_input': 'Examples:\nelevation->depression\nlimitless->limited\nluxurious->basic\nimmature->mature\nidiot->genius\nunavailable->available\ninner->outer\nadolescent->elderly\nnominal->real\nblend->stand out\nrelevant->irrelevant\nsober->drunk\nbrethren->sistren\noutfield->infield\nentire->partial\nwise->foolish\nrainy->sunny\npermit->prohibit\npopular->unpopular\nviable->

# Load Model

In [7]:
llm = LanguageModel(model, device_map="auto", dispatch=True, dtype=torch.bfloat16)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# Model Functions

In [8]:
def get_representations(samples):
    representations = []
    for sample in tqdm(samples):
        with torch.no_grad():
            with llm.trace(sample["context"]) as tracer:
                # tokens = llm.tokenizer.tokenize(sample["context"])
                # last_token_target_idx = len(tokens) - 1 - tokens[::-1].index(token)

                partial_representation = []
                for i in range(len(llm.model.layers)):
                    representation = (
                        llm.model.layers[i]
                        .output[0].mean(dim=0)
                        .save()
                    )
                    partial_representation.append(representation)

                representations.append(partial_representation)

    return representations


In [18]:
def generate_text_with_representations(
    samples, representations: List[torch.Tensor], token_changed: str
):
    for sample in samples:
        sample["0shot_response"] = []

        # Get the index of the last separator token
        tokens = llm.tokenizer.convert_ids_to_tokens(
            llm.tokenizer(sample["0shot_input"])["input_ids"]
        )
        last_token_changed_idx = len(tokens) - 1 - tokens[::-1].index(token_changed)

        for layer_idx in tqdm(range(len(representations))):
            with llm.generate(
                sample["0shot_input"],
                max_new_tokens=5,
                do_sample=False,
            ) as generator:
                llm.model.layers[layer_idx].output[0][last_token_changed_idx, :] += (
                    representations[layer_idx]
                )

                generation = llm.generator.output.save()

            sample["0shot_response"].append(
                llm.tokenizer.decode(generation[0], skip_special_tokens=True)
            )


In [10]:
def generate_text(samples):
    vllm = LLM("google/gemma-3-4b-pt", dtype=torch.bfloat16, gpu_memory_utilization=0.5)

    inputs = [sample["nshot_input"] for sample in samples]
    sampling_params = SamplingParams(
        temperature=0, max_tokens=150, n=1, stop=[seperator]
    )
    outputs = vllm.generate(inputs, sampling_params)
    for i, output in enumerate(outputs):
        samples[i]["nshot_response"] = output.outputs[0].text

# Metrics

In [11]:
bleu = BLEU(tokenize="flores200")
chrf = CHRF(word_order=2)


def evaluate(samples: List[dict]) -> dict:
    output = [s["output"] for s in samples]
    reference = [[s["reference"]] for s in samples]

    return {
        "bleu": bleu.corpus_score(output, reference).score,
        "chrf": chrf.corpus_score(output, reference).score,
    }

# Experiments !!!

In [12]:
llm.model.layers

ModuleList(
  (0-31): 32 x LlamaDecoderLayer(
    (self_attn): LlamaAttention(
      (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
      (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
      (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
      (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
    )
    (mlp): LlamaMLP(
      (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
      (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
      (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
      (act_fn): SiLUActivation()
    )
    (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
    (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
  )
)

In [13]:
representations = get_representations(samples)
len(representations), len(representations[0]), representations[0][0].shape


100%|██████████| 79/79 [00:02<00:00, 30.49it/s]


(79, 32, torch.Size([4096]))

In [14]:
[samples[0], samples[1], samples[2], samples[3]]

[{'context': 'Examples:\nelevation->depression\nlimitless->limited\nluxurious->basic\nimmature->mature\nidiot->genius\nunavailable->available\ninner->outer\nadolescent->elderly\nnominal->real\nblend->stand out\nrelevant->irrelevant\nsober->drunk\nbrethren->sistren\noutfield->infield\nentire->partial\nwise->foolish\nrainy->sunny\npermit->prohibit\npopular->unpopular\nviable->unviable\nsupporter->opponent\nescalate->deescalate\nmobile->stationary\nclosing->opening\nskew->straight\npowerless->powerful\naccompany->leave alone\nsevere->mild\ngraduating->enrolling\n',
  'query': 'commence',
  'reference': 'conclude',
  'nshot_input': 'Examples:\nelevation->depression\nlimitless->limited\nluxurious->basic\nimmature->mature\nidiot->genius\nunavailable->available\ninner->outer\nadolescent->elderly\nnominal->real\nblend->stand out\nrelevant->irrelevant\nsober->drunk\nbrethren->sistren\noutfield->infield\nentire->partial\nwise->foolish\nrainy->sunny\npermit->prohibit\npopular->unpopular\nviable->

In [15]:
# Merge all representations
merged_representations = []
for layer_idx in range(len(representations[0])):
    layer_representations = torch.cat(
        [representation[layer_idx] for representation in representations], dim=0
    )
    merged_representations.append(layer_representations.mean(dim=0, keepdim=True))

len(merged_representations), len(merged_representations[0]), merged_representations[0].shape

(32, 1, torch.Size([1]))

In [19]:
generate_text_with_representations([samples[0], samples[1], samples[2], samples[3]], merged_representations, transisition_token)

100%|██████████| 32/32 [00:03<00:00,  9.08it/s]


In [20]:
[samples[0]['0shot_response'], samples[1]['0shot_response'], samples[2]['0shot_response'], samples[3]['0shot_response']]

[['Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\ncommence->commenced\ncom',
  'Examples:\nco

In [156]:
generate_text([samples[1]])

`torch_dtype` is deprecated! Use `dtype` instead!
We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


(EngineCore_DP0 pid=1802715) EngineCore failed to start.
(EngineCore_DP0 pid=1802715) Traceback (most recent call last):
(EngineCore_DP0 pid=1802715)   File "/home/tlasnier/mitra/.venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 699, in run_engine_core
(EngineCore_DP0 pid=1802715)     engine_core = EngineCoreProc(*args, **kwargs)
(EngineCore_DP0 pid=1802715)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore_DP0 pid=1802715)   File "/home/tlasnier/mitra/.venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 498, in __init__
(EngineCore_DP0 pid=1802715)     super().__init__(vllm_config, executor_class, log_stats,
(EngineCore_DP0 pid=1802715)   File "/home/tlasnier/mitra/.venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 83, in __init__
(EngineCore_DP0 pid=1802715)     self.model_executor = executor_class(vllm_config)
(EngineCore_DP0 pid=1802715)                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore_DP0 pid=1802715)   File "/h

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [ ]:
samples[1]["nshot_response"]

'nonviable'

# Test

In [156]:
context = "The capital of France is"
test_representations = []

tokens = llm.tokenizer.tokenize(context)
last_token_target_idx = len(tokens) - 1 - tokens[::-1].index("▁is")

with llm.trace(context) as tracer:
    for layer_idx in tqdm(range(len(llm.model.layers))):
        representation = (
            llm.model.layers[layer_idx]
            .output[0][last_token_target_idx, :]
            .save()
        )
        test_representations.append(representation)

    last_layer_representation = (
        llm.model.layers[-1].output[0][last_token_target_idx, :].save()
    )
len(test_representations), test_representations[0].shape

100%|██████████| 32/32 [00:00<00:00, 634.11it/s]


(32, torch.Size([4096]))

In [158]:
altered_context = "The name of capital of Spain is"
tokens = llm.tokenizer.tokenize(altered_context)
last_token_target_idx = len(tokens) - 1 - tokens[::-1].index("▁is")

generations = []
for layer_idx in tqdm(range(len(test_representations))):
    with llm.generate(
        altered_context,
        max_new_tokens=10,
        do_sample=False,
    ) as generator:
        llm.model.layers[layer_idx].output[0][last_token_target_idx, :] = (
            test_representations[layer_idx]
        )

        generation = llm.generator.output.save()

    generations.append(llm.tokenizer.decode(generation[0], skip_special_tokens=True))

generations

100%|██████████| 32/32 [00:07<00:00,  4.12it/s]


['The name of capital of Spain is Paris.\nThe name of capital of France is',
 'The name of capital of Spain is Paris.\nThe name of capital of France is',
 'The name of capital of Spain is Paris.\nThe name of capital of France is',
 'The name of capital of Spain is Paris.\nThe name of capital of France is',
 'The name of capital of Spain is Paris.\nThe name of capital of France is',
 'The name of capital of Spain is Paris.\nThe name of capital of France is',
 'The name of capital of Spain is Paris.\nThe name of capital of France is',
 'The name of capital of Spain is Paris.\nThe name of capital of France is',
 'The name of capital of Spain is Paris.\nThe name of capital of France is',
 'The name of capital of Spain is Paris.\nThe name of capital of France is',
 'The name of capital of Spain is Paris.\nThe name of capital of France is',
 'The name of capital of Spain is Paris.\nThe name of capital of France is',
 'The name of capital of Spain is Paris.\nThe name of capital of France is',

In [91]:
llm

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
   

In [186]:
with llm.generate("Madison square garden is located in the city of New", max_new_tokens=3) as tracer:
    embeddings = llm.model.layers[-1].output.save()
    original = llm.generator.output.save()

print(llm.tokenizer.batch_decode(original))

with llm.generate("_ _ _ _ _ _ _ _ _ _", max_new_tokens=3, do_sample=False) as tracer:
    # since this is a separate run, we don't have to use barriers
    llm.model.layers[-1].output = embeddings
    intervened = llm.generator.output.save()

print(llm.tokenizer.batch_decode(intervened))

['<s> Madison square garden is located in the city of New York. Mad']
['<s> _ _ _ _ _ _ _ _ _ _ Yorkshire\n']


In [ ]:
def calculate_h_and_intervene(
    model: LanguageModel,
    dataset: ICLDataset,
    zero_shot_dataset: ICLDataset,
    layer: int,
) -> tuple[list[str], list[str]]:
    """
    Extracts the vector `h`, intervenes by adding `h` to the residual stream of a set of generated zero-shot prompts,
    all within the same forward pass. Returns the completions from this intervention.

    Inputs:
        model: LanguageModel
            the model we're using to generate completions
        dataset: ICLDataset
            the dataset of clean prompts from which we'll extract the `h`-vector
        zero_shot_dataset: ICLDataset
            the dataset of zero-shot prompts which we'll intervene on, using the `h`-vector
        layer: int
            the layer we'll be extracting the `h`-vector from

    Returns:
        completions_zero_shot: list[str]
            list of string completions for the zero-shot prompts, without intervention
        completions_intervention: list[str]
            list of string completions for the zero-shot prompts, with h-intervention
    """
    with model.trace() as tracer:
        with tracer.invoke(dataset.prompts):
            h = model.transformer.h[layer].output[0][:, -1].mean(dim=0)
        
        with tracer.invoke(zero_shot_dataset.prompts):
            clean_tokens = model.lm_head.output[:, -1].argmax(dim=-1).save()

        with tracer.invoke(zero_shot_dataset.prompts):
            hidden = model.transformer.h[layer].output[0]
            hidden[:, -1] += h
            intervene_tokens = model.lm_head.output[:, -1].argmax(dim=-1).save()

    completions_zero_shot = tokenizer.batch_decode(clean_tokens)
    completions_intervention = tokenizer.batch_decode(intervene_tokens)
    return completions_zero_shot, completions_intervention